In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
path = "/content/drive/MyDrive/fin-research-agent/finetune/train.jsonl"
print(os.path.exists(path))  # should print True

True


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [5]:
ALPACA_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_example(example):
    text = ALPACA_TEMPLATE.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )
    return {"text": text}

In [6]:
from datasets import load_dataset

full_dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/fin-research-agent/finetune/train.jsonl",
    split="train"
)

toy_dataset = full_dataset.select(range(25))
toy_dataset = toy_dataset.map(format_example)

print("Sample formatted row:")
print(toy_dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Sample formatted row:
### Instruction:
You are a financial report quality judge. Score the following analyst
report using this rubric. Each criterion is scored 1-10. Overall score is
the average of the three, unless one criterion scores 1-3, in which case
overall should not exceed 5.

1. GROUNDING — Does every claim trace back to explicit language in its
cited chunk (not implied, not adjacent, not a reasonable inference)?
1-3: Claims reference chunks that don't support them, wrong-section
citations, or fabricated/garbled facts presented as real.
4-7: Right area but overstates, adds unstated framing, or blurs a risk
into a positive without contradicting the source.
8-10: Every claim directly traceable to explicit chunk language.

2. COMPLETENESS — Does the report cover what actually matters in the
filing, without padding or omitting a major risk/strength?
1-3: Missing an obviously major risk/opportunity, or generic boilerplate
points.
4-7: Real substance but thin, or one boilerplate poi

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.8.22 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    train_dataset=toy_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs_toy",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/25 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.085980
2,2.131779
3,2.351847
4,2.090561
5,2.198019
6,2.087605
7,1.936324
8,1.960347
9,2.021891
10,2.121386


TrainOutput(global_step=12, training_loss=2.073836545149485, metrics={'train_runtime': 98.8002, 'train_samples_per_second': 0.759, 'train_steps_per_second': 0.121, 'total_flos': 1282013134848000.0, 'train_loss': 2.073836545149485, 'epoch': 3.0})

In [9]:
model.save_pretrained("toy_adapter")
tokenizer.save_pretrained("toy_adapter")

Unsloth: Restored added_tokens_decoder metadata in toy_adapter/tokenizer_config.json.


('toy_adapter/tokenizer_config.json',
 'toy_adapter/chat_template.jinja',
 'toy_adapter/tokenizer.json')

In [10]:
FastLanguageModel.for_inference(model)

test_prompt = ALPACA_TEMPLATE.format(
    instruction=full_dataset[5]["instruction"],
    input=full_dataset[5]["input"],
    output="",
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Unsloth: Input IDs of shape torch.Size([1, 3724]) with length 3724 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


### Instruction:
You are a financial report quality judge. Score the following analyst
report using this rubric. Each criterion is scored 1-10. Overall score is
the average of the three, unless one criterion scores 1-3, in which case
overall should not exceed 5.

1. GROUNDING — Does every claim trace back to explicit language in its
cited chunk (not implied, not adjacent, not a reasonable inference)?
1-3: Claims reference chunks that don't support them, wrong-section
citations, or fabricated/garbled facts presented as real.
4-7: Right area but overstates, adds unstated framing, or blurs a risk
into a positive without contradicting the source.
8-10: Every claim directly traceable to explicit chunk language.

2. COMPLETENESS — Does the report cover what actually matters in the
filing, without padding or omitting a major risk/strength?
1-3: Missing an obviously major risk/opportunity, or generic boilerplate
points.
4-7: Real substance but thin, or one boilerplate point mixed in.
8-10: Eve

In [11]:
!zip -r toy_adapter.zip toy_adapter
from google.colab import files
files.download("toy_adapter.zip")

  adding: toy_adapter/ (stored 0%)
  adding: toy_adapter/tokenizer_config.json (deflated 89%)
  adding: toy_adapter/README.md (deflated 65%)
  adding: toy_adapter/adapter_model.safetensors (deflated 9%)
  adding: toy_adapter/tokenizer.json (deflated 81%)
  adding: toy_adapter/adapter_config.json (deflated 59%)
  adding: toy_adapter/chat_template.jinja (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>